In [1]:
import pandas as pd
from tqdm import tqdm
import os
import datetime as dt
import sys
import glob

In [2]:
sys.path.append("/data/workspace/libs/pytools")
db_dir = "/data/workspace/Data/DataBase"
from mfrt.research_tools import DBTool

In [3]:
from mfrt.opt_backtest import OptBacktest
from mfrt.eval_netvalue import EvalNetValue
opt_bt=OptBacktest(db_dir)
eval_NV=EvalNetValue(db_dir)

from mfrt.eval_factor import EvalFactor
eval_F = EvalFactor(db_dir)

check_Barra_list=('BETA', 'MOMENTUM', 'SIZE', 'RESVOL', 'BTOP', 'LIQUIDTY',  'SPRET', 'SIZENL', 'GROWTH', 'LEVERAGE')
def test_factor(factor: pd.DataFrame, 
                start_dt, 
                end_dt, 
                display: bool=False, 
                if_trade_tmr: bool=True, 
                specific_return: bool=True,
                Universe=None,
                return_type='Vwap') -> dict:
    F_res=eval_F.quick_test(factor,start_dt=start_dt,end_dt=end_dt,Group_Num=10,Horizon=10,Return_Type=return_type,
                        Specific_Return=specific_return,Universe=Universe,check_Barra_list=check_Barra_list,
                        extraExp_check_dict=None,if_trade_tmr=if_trade_tmr, display=display)
    
    return F_res

In [33]:
df = pd.read_parquet("/home/intern_fjq_2026/Projects/chinese-wwm-roberta/artifacts/continuous_label_layer_probe/runs/static_fy0_csi300_2025h1_v1/csi300_rank_ic/csi300_stock_day_factors.parquet")
factor = df[["trading_date", "symbol", "prediction_layer_6"]].sort_values(by=["trading_date", "symbol"])
factor["trading_date"] = pd.to_datetime(factor["trading_date"], unit="D")

In [34]:
factor_agg = df.groupby(["trading_date", "symbol"])["prediction_layer_6"].mean().reset_index()

In [35]:
factor_agg

,trading_date,symbol,prediction_layer_6
0,2023-01-03,000157,0.065584
1,2023-01-03,000301,0.056120
2,2023-01-03,000425,0.048839
3,2023-01-03,000568,-0.050425
4,2023-01-03,000596,-0.053765
...,...,...,...
30678,2025-08-01,601728,0.010559
30679,2025-08-01,601816,-0.185731
30680,2025-08-01,601888,-0.131716
30681,2025-08-01,603369,-0.194835


In [36]:
factor_wide = factor_agg.set_index(['trading_date', 'symbol'])['prediction_layer_6'].unstack()
factor_wide.index.name = 'date' 

In [37]:
factor_wide

symbol,000001,000002,000063,000069,000100,000157,000166,000301,000333,000338,...,688256,688271,688303,688363,688396,688472,688506,688561,688599,688981
date,,,,,,,,,,,,,,,,,,,,,
2023-01-03,NaN,NaN,NaN,NaN,NaN,0.065584,NaN,0.05612,NaN,NaN,...,NaN,NaN,0.049659,NaN,NaN,NaN,NaN,NaN,0.049659,NaN
2023-01-04,NaN,-0.015747,-0.156109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-05,NaN,-0.038305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.033062,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.041029,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-07-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
counts = ((factor_wide != 0) & factor_wide.notna()).sum(axis=1)
counts

date
2023-01-03    82
2023-01-04    36
2023-01-05    27
2023-01-06    19
2023-01-09    64
              ..
2025-07-28    31
2025-07-29    13
2025-07-30    15
2025-07-31    24
2025-08-01    22
Length: 620, dtype: int64

In [40]:
# res = test_factor(
#             factor_wide.dropna(how="all", axis=0),
#             start_dt='2023.01.01',
#             end_dt='2024.12.31',
#             display=True,
#             if_trade_tmr=True,
#             specific_return=False,
#             Universe=,
#             return_type="Vwap",
#         )
# res